# 03 - 特徵萃取（驗證）

批次特徵萃取由 `src/run_feature_extraction.py` 執行：
```bash
cd src && python run_feature_extraction.py
```

萃取內容：
- **MFCC**：13 + delta + delta-delta = 39 維，shape (n_frames, 39)
- **Mel-spectrogram**：n_mels=128, hop_length=512, shape (128, 94)
- 萃取時做 3 秒 pad/truncate（取中間段）

本 notebook 用於**驗證萃取結果與視覺化**。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
MFCC_DIR = PROJECT_ROOT / "data" / "features" / "mfcc"
MELSPEC_DIR = PROJECT_ROOT / "data" / "features" / "melspec"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

df = pd.read_csv(PROJECT_ROOT / "data" / "metadata.csv")
print(f"樣本數：{len(df):,}")
print(f"欄位：{df.columns.tolist()}")

# 確認特徵欄位存在
assert "mfcc_path" in df.columns, "請先執行 src/run_feature_extraction.py"
assert "melspec_path" in df.columns, "請先執行 src/run_feature_extraction.py"
print("\n特徵欄位確認 OK")

## 1. Shape 驗證

In [ ]:
# 抽樣驗證特徵 shape
print("特徵 shape 抽樣驗證：\n")
sample = df.sample(10, random_state=42)
for _, row in sample.iterrows():
    mfcc = np.load(PROJECT_ROOT / row["mfcc_path"])
    melspec = np.load(PROJECT_ROOT / row["melspec_path"])
    print(f"  [{row['dataset']:10s}] {row['emotion']:8s} "
          f"MFCC={mfcc.shape}, Mel-spec={melspec.shape}")

# 全量抽樣 100 筆確認一致性
print("\n全量 shape 檢查（抽樣 100 筆）...")
sample_large = df.sample(min(100, len(df)), random_state=0)
mfcc_shapes = set()
melspec_shapes = set()
for _, row in sample_large.iterrows():
    mfcc_shapes.add(np.load(PROJECT_ROOT / row["mfcc_path"]).shape)
    melspec_shapes.add(np.load(PROJECT_ROOT / row["melspec_path"]).shape)

print(f"  MFCC unique shapes: {mfcc_shapes}")
print(f"  Mel-spec unique shapes: {melspec_shapes}")
if len(mfcc_shapes) == 1 and len(melspec_shapes) == 1:
    print("\n[OK] 所有特徵 shape 一致")
else:
    print("\n[WARNING] 特徵 shape 不一致！")

## 2. Mel-spectrogram 視覺化抽檢

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

emotions = sorted(df["emotion"].unique())
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"{e}" for e in emotions],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for i, emotion in enumerate(emotions):
    row = i // 3 + 1
    col = i % 3 + 1
    sample_row = df[df["emotion"] == emotion].sample(1, random_state=42).iloc[0]
    mel_spec = np.load(PROJECT_ROOT / sample_row["melspec_path"])
    fig.add_trace(
        go.Heatmap(z=mel_spec, colorscale="Viridis", showscale=(i == 0)),
        row=row, col=col,
    )

fig.update_layout(
    title_text="各情緒 Mel-spectrogram 範例（128x94, 3秒）",
    height=500, width=900, template="plotly_white",
)
fig.write_html(FIGURES_DIR / "06_melspec_examples.html")
fig.write_image(FIGURES_DIR / "06_melspec_examples.png", width=900, height=500, scale=2)
fig.show()

## 3. 特徵檔案統計

In [ ]:
def dir_size_mb(directory: Path) -> tuple[float, int]:
    files = list(directory.glob("*.npy"))
    total = sum(f.stat().st_size for f in files)
    return total / 1e6, len(files)

mfcc_mb, mfcc_count = dir_size_mb(MFCC_DIR)
melspec_mb, melspec_count = dir_size_mb(MELSPEC_DIR)

print("特徵檔案統計：")
print(f"  MFCC:     {mfcc_count:>6,} 檔案, {mfcc_mb:>8.1f} MB")
print(f"  Mel-spec: {melspec_count:>6,} 檔案, {melspec_mb:>8.1f} MB")
print(f"  合計:     {mfcc_count + melspec_count:>6,} 檔案, {mfcc_mb + melspec_mb:>8.1f} MB")